# Advanced C# for .NET Core Architects\n\nThis notebook goes deeper into C# building blocks that are heavily used in modern .NET Core apps:\n- Delegates and events\n- Interfaces vs abstract classes (for DI and architecture)\n- `IEnumerable<T>` vs `IQueryable<T>`\n- Attributes and reflection (annotations)\n\nRun each code cell, then modify it to see how behavior changes.\n

## 1. Delegates and Events\n\nDelegates are types that represent methods. Events build on delegates to implement the observer pattern.\nIn real apps, events are used for domain events, UI events, and background notifications.\n

In [ ]:
// A delegate type: any method that matches (string message) can be attached.\ndelegate void NotificationHandler(string message);\n\nclass Notifier\n{\n    public event NotificationHandler? OnNotify;\n\n    public void DoWork()\n    {\n        // ... some work\n        OnNotify?.Invoke("Work completed.");\n    }\n}\n\nvoid Logger(string message) => Console.WriteLine($"LOG: {message}");\n\nvar notifier = new Notifier();\nnotifier.OnNotify += Logger;\nnotifier.DoWork();\n

## 2. Interfaces vs Abstract Classes (for DI)\n\nInterfaces define *contracts*; abstract classes can provide a partial default implementation.\nIn ASP.NET Core you typically inject interfaces into constructors to keep code testable and flexible.\n

In [ ]:
interface IEmailSender\n{\n    void Send(string to, string subject, string body);\n}\n\nclass SmtpEmailSender : IEmailSender\n{\n    public void Send(string to, string subject, string body)\n    {\n        Console.WriteLine($"Sending email to {to}: {subject}");\n    }\n}\n\nabstract class BaseController\n{\n    protected readonly IEmailSender EmailSender;\n    protected BaseController(IEmailSender emailSender) => EmailSender = emailSender;\n}\n\nclass UserController : BaseController\n{\n    public UserController(IEmailSender sender) : base(sender) { }\n\n    public void Register(string email)\n    {\n        // domain logic ...\n        EmailSender.Send(email, "Welcome", "Thanks for registering.");\n    }\n}\n\nIEmailSender emailSender = new SmtpEmailSender();\nvar controller = new UserController(emailSender);\ncontroller.Register("user@example.com");\n

## 3. `IEnumerable<T>` vs `IQueryable<T>`\n\n- `IEnumerable<T>`: in-memory collections, LINQ runs in .NET (good for lists, arrays).\n- `IQueryable<T>`: query is *translated* (e.g., to SQL in EF Core) and executed by an external provider.\n\nKey idea: with `IQueryable<T>` **defer materialization** and let the database do the heavy lifting.\n

In [ ]:
using System.Collections.Generic;\nusing System.Linq;\n\nList<int> numbers = new() { 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 };\nIEnumerable<int> evenEnumerable = numbers.Where(n => n % 2 == 0);\nConsole.WriteLine("IEnumerable result: " + string.Join(", ", evenEnumerable));\n\nIQueryable<int> queryable = numbers.AsQueryable();\nIQueryable<int> evenQueryable = queryable.Where(n => n % 2 == 0);\nConsole.WriteLine("IQueryable provider: " + evenQueryable.Provider.GetType().Name);\nConsole.WriteLine("IQueryable expression: " + evenQueryable.Expression);\nConsole.WriteLine("IQueryable materialized: " + string.Join(", ", evenQueryable.ToList()));\n

## 4. Attributes and Reflection\n\nAttributes are metadata you attach to types and members (similar to annotations).\nFrameworks (ASP.NET Core, EF Core, testing frameworks) read attributes via reflection to change behavior.\n

In [ ]:
using System;\nusing System.Reflection;\n\n[AttributeUsage(AttributeTargets.Class | AttributeTargets.Method)]\nclass AuditAttribute : Attribute\n{\n    public string Action { get; }\n    public AuditAttribute(string action) => Action = action;\n}\n\n[Audit("CreateOrder")]\nclass OrderService\n{\n    [Audit("PlaceOrder")]\n    public void PlaceOrder() { }\n}\n\nvar type = typeof(OrderService);\nvar classAudit = type.GetCustomAttribute<AuditAttribute>();\nConsole.WriteLine($"Class Audit Action: {classAudit?.Action}");\n\nvar method = type.GetMethod("PlaceOrder");\nvar methodAudit = method?.GetCustomAttribute<AuditAttribute>();\nConsole.WriteLine($"Method Audit Action: {methodAudit?.Action}");\n